In [ ]:
import pandas as pd
import calendar
import re

indent_file = r"D:/PPC Plan/Monthly Indent/Monthly Indent.xlsx"
bom_file    = r"D:/Tushar/main_with_subs_only.xlsx"

# ───────────────────────────────────────────────
# Read files
# ───────────────────────────────────────────────
try:
    indent_df = pd.read_excel(indent_file)
    bom_df    = pd.read_excel(bom_file)
except Exception as e:
    print("File read error:", e)
    input("Press Enter to exit...")
    exit()

# Clean column names
for df in [indent_df, bom_df]:
    df.columns = df.columns.str.strip()

# ───────────────────────────────────────────────
# Safe string conversion (prevents float NaN errors)
# ───────────────────────────────────────────────
def safe_to_str(series):
    # Convert to string, handle NaN, remove extra spaces
    return series.astype(str).replace('nan', '').str.strip()

bom_df['Sub_Label']  = safe_to_str(bom_df['Sub_Label'])
bom_df['Main_Label'] = safe_to_str(bom_df['Main_Label'])

# ───────────────────────────────────────────────
# Auto-detect part code column in indent file
# ───────────────────────────────────────────────
possible_part_cols = [
    c for c in indent_df.columns
    if any(k in c.lower() for k in ['part', 'code', 'fg', 'model', 'item', 'number', 'no.', 'switch'])
]
possible_qty_cols  = [
    c for c in indent_df.columns
    if any(k in c.lower() for k in ['qty', 'indent', 'plan', 'demand', 'req', 'quantity'])
]

print("Possible part columns in indent file:", possible_part_cols)
print("Possible qty columns in indent file :", possible_qty_cols)

# ───────────────────────────────────────────────
# Month / days detection
# ───────────────────────────────────────────────
pattern = re.compile(r"([A-Za-z]{3})'(\d{2})", re.I)
month_cols = [c for c in indent_df.columns if pattern.search(str(c))]

if not month_cols:
    print("No month columns like 'Jan'25' found.")
    print("All columns:", list(indent_df.columns))
    input("Press Enter to exit...")
    exit()

latest_col = max(month_cols, key=str)  # last one usually = latest
print("\nDetected latest month column →", latest_col)

match = pattern.search(latest_col)
if not match:
    print("Cannot parse month from:", latest_col)
    exit()

month_str = match.group(1).title()
year = 2000 + int(match.group(2))
month_num = list(calendar.month_abbr).index(month_str)
days_in_month = calendar.monthrange(year, month_num)[1]
print(f"→ {month_str} {year} has {days_in_month} days\n")

# ───────────────────────────────────────────────
# Prepare indent data
# ───────────────────────────────────────────────
part_col = 'Part number'  # <--- CHANGE THIS if the auto-detect shows different name

if part_col not in indent_df.columns and possible_part_cols:
    part_col = possible_part_cols[0]   # take first guess if default not found
    print(f"Using detected part column: {part_col}")

if part_col not in indent_df.columns:
    print(f"Column '{part_col}' not found. Available columns:", list(indent_df.columns))
    input("Press Enter to exit...")
    exit()

indent_df[part_col] = safe_to_str(indent_df[part_col])

indent_df = indent_df[[part_col, latest_col]].dropna(subset=[latest_col])
indent_df[latest_col] = pd.to_numeric(indent_df[latest_col], errors='coerce')
indent_df = indent_df.dropna(subset=[latest_col])

indent_df = indent_df.rename(columns={part_col: 'Switch', latest_col: 'Monthly_Qty'})
indent_df['Daily_Qty'] = indent_df['Monthly_Qty'] / days_in_month

print("Indent summary (first 8 rows):")
print(indent_df.head(8))
print(f"→ {len(indent_df)} switch types with plan\n")

# ───────────────────────────────────────────────
# Prepare BOM
# ───────────────────────────────────────────────
bom_df = bom_df[['Main_Label', 'Sub_Label', 'Sub_Count']].copy()
bom_df = bom_df.rename(columns={'Main_Label': 'Child', 'Sub_Label': 'Switch', 'Sub_Count': 'Usage_Qty'})

bom_df['Switch'] = safe_to_str(bom_df['Switch'])
bom_df['Child']  = safe_to_str(bom_df['Child'])

print(f"BOM has {len(bom_df):,} lines\n")

# ───────────────────────────────────────────────
# Merge
# ───────────────────────────────────────────────
merged = pd.merge(
    bom_df,
    indent_df[['Switch', 'Daily_Qty']],
    on='Switch',
    how='inner'
)

print(f"After merge: {len(merged):,} matching rows")

if len(merged) == 0:
    print("\nSample BOM switch codes (first 10):")
    print(bom_df['Switch'].unique()[:10].tolist())
    print("\nSample Indent switch codes (first 10):")
    print(indent_df['Switch'].unique()[:10].tolist())
    print("\n→ No matches → formatting mismatch is almost certainly the issue")
    input("Press Enter to exit...")
    exit()

merged['Daily_Child_Need'] = merged['Daily_Qty'] * merged['Usage_Qty']

# Aggregate per child
result = merged.groupby('Child', as_index=False)['Daily_Child_Need'].sum()
result = result.rename(columns={'Daily_Child_Need': 'Daily_Qty'})
result['Two_Day_Qty'] = result['Daily_Qty'] * 2

result = result.sort_values('Two_Day_Qty', ascending=False)
result[['Daily_Qty', 'Two_Day_Qty']] = result[['Daily_Qty', 'Two_Day_Qty']].round(2)

print("\nTop 15 results:")
print(result.head(15))

# Save
output_file = "Two_Day_Child_Qty.xlsx"
result.to_excel(output_file, index=False)
print(f"\nSaved to: {output_file}")